## Set up Data

#### Mount drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#### Unzip the Images

In [ ]:
import zipfile, os

CHALLENGE_DIR = "/content/drive/MyDrive/AML"
ZIP_PATH      = os.path.join(CHALLENGE_DIR, "dataset.zip")
IMAGES_DIR    = "/content/images"   # extracted here (fast local SSD, not Drive)

if not os.path.exists(IMAGES_DIR):
    print("Extracting images...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(IMAGES_DIR)
    print(f"Done — extracted to {IMAGES_DIR}")
else:
    print(f"Already extracted: {IMAGES_DIR}")

# Peek at what's inside
sample_files = os.listdir(os.path.join(IMAGES_DIR,"images"))[:5]
print(f"Total images : {len(os.listdir(os.path.join(IMAGES_DIR,"images")))}")
print(f"Sample names : {sample_files}")
IMAGES_DIR = os.path.join(IMAGES_DIR,"images")

#### Load the Training Data

In [ ]:
import pandas as pd

train_df = pd.read_csv(os.path.join(CHALLENGE_DIR, "train.csv"))
test_df  = pd.read_csv(os.path.join(CHALLENGE_DIR, "test.csv"))

# Map image_id → full path
def img_path(image_id):
    # Try common extensions; adjust if your images have a fixed extension
    for ext in ["", ".jpg", ".jpeg", ".png", ".webp"]:
        p = os.path.join(IMAGES_DIR, f"{image_id}{ext}")
        if os.path.exists(p):
            return p
    return None

train_df["path"] = train_df["image_id"].apply(img_path)
test_df["path"]  = test_df["image_id"].apply(img_path)

print(f"Train rows   : {len(train_df)}")
print(f"Test  rows   : {len(test_df)}")
print("\nLabel distribution:")
print(train_df["ground_truth"].value_counts().rename({0: "authentic", 1: "ai_generated"}))


## Set Up EfficientNetV2

In [ ]:
model_name = "tf_efficientnetv2_s.in21k"
FOLD_TO_RUN = 4

### Load Features

In [ ]:
import numpy as np

# TRAIN_FEATURES_PATH = os.path.join(CHALLENGE_DIR, f"features/{model_name}_train_features.npy")
# TRAIN_LABELS_PATH = os.path.join(CHALLENGE_DIR, f"features/{model_name}_train_labels.npy")
# VAL_FEATURES_PATH = os.path.join(CHALLENGE_DIR, f"features/{model_name}_val_features.npy")
# VAL_LABELS_PATH = os.path.join(CHALLENGE_DIR, f"features/{model_name}_val_labels.npy")
# TEST_FEATURES_PATH = os.path.join(CHALLENGE_DIR, f"features/{model_name}_test_features.npy")
# TEST_LABELS_PATH = os.path.join(CHALLENGE_DIR, f"features/{model_name}_test_labels.npy")

TRAIN_FEATURES_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_train_features.npy")
TRAIN_LABELS_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_train_labels.npy")
VAL_FEATURES_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_val_features.npy")
VAL_LABELS_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_val_labels.npy")
TEST_FEATURES_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_test_features.npy")
TEST_LABELS_PATH = os.path.join(CHALLENGE_DIR, f"features/try{FOLD_TO_RUN + 1}_test_labels.npy")

X_train = np.load(TRAIN_FEATURES_PATH)
X_val = np.load(VAL_FEATURES_PATH)
X_test = np.load(TEST_FEATURES_PATH)
y_train = np.load(TRAIN_LABELS_PATH)
y_val = np.load(VAL_LABELS_PATH)
y_test = np.load(TEST_LABELS_PATH)

print(X_train.shape, X_val.shape, X_test.shape, y_train.shape, y_val.shape, y_test.shape)

## XGBoost

In [ ]:
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, roc_auc_score, roc_curve
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

#### Find best param

In [ ]:
classifier = XGBClassifier(tree_method="hist", device="cuda", random_state=42)

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 500, 1000],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 9],
    'min_child_weight': [1, 3, 5],
}

X_combined = np.vstack((X_train, X_val))
y_combined = np.concatenate((y_train, y_val))
split_index = np.concatenate(((-1 * np.ones(len(X_train))), (0 * np.ones(len(X_val)))))
pds = PredefinedSplit(split_index)

grid = GridSearchCV(classifier, param_grid, cv=pds, scoring='f1')
grid.fit(X_combined, y_combined, eval_set=[(X_val, y_val)], verbose=False)

print(grid.best_params_, grid.best_score_)

#### Straight

In [ ]:
classifier = XGBClassifier(random_state=42, learning_rate=0.1, max_depth=3, min_child_weight=3, n_estimators=1000)
classifier.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

#### Prediction and report

In [ ]:
y_pred = classifier.predict(X_test)
y_probs = classifier.predict_proba(X_test)[:, 1]

print("\nXGBoost Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Authentic", "AI-Generated"]))
print(f"F1: {f1_score(y_test, y_pred):.3f}")

In [ ]:
y_test_probs_xgb = classifier.predict_proba(X_test)[:, 1]

auc_score_xgb = roc_auc_score(y_test, y_test_probs_xgb)
fpr, tpr, _ = roc_curve(y_test, y_test_probs_xgb)

plt.figure(figsize=(3.5, 3))
plt.plot(fpr, tpr, color='forestgreen', lw=1.5, label=f'XGBoost (AUC={auc_score_xgb:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Random')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=9)
plt.ylabel('True Positive Rate', fontsize=9)
plt.xticks(fontsize=8)
plt.yticks(fontsize=8)
plt.legend(loc="lower right", fontsize=8)
plt.grid(True, alpha=0.2)

plt.savefig('roc_curve_xgb.pdf', format='pdf', dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Real', 'AI'])
disp.plot(cmap='Blues', ax=ax)
plt.title('XGB Confusion Matrix')
plt.show()